In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import os, sys


def get_dir_n_levels_up(path, n):
    # Go up n levels from the given path
    for _ in range(n):
        path = os.path.dirname(path)
    return path

proj_root = get_dir_n_levels_up(os.path.abspath("__file__"), 3)
sys.path.append(proj_root)

In [ ]:
# =========================================================
# Multi-seed experiment (10 seeds) — collect metrics only
# =========================================================

import matplotlib.pyplot as plt

# Your identification + control pieces (from your repo)

from opinion_dynamics.experiments.online import run_multi_seed_experiment
from opinion_dynamics.experiments.plots import plot_impulse_node_trajectories, show_matrix_with_cell_grid
from opinion_dynamics.experiments.rollouts import rollout_with_policy_intermediate
from opinion_dynamics.experiments.metrics import graph_sanity


# =========================================================
# Run it
# =========================================================
df = run_multi_seed_experiment(
    seeds=range(10),
    B_campaign=1.0,
    num_campaigns_total=5,
    lr=1e-3,
    l2_lambda=0.0,
    device="cpu",
    update_A_each_campaign=True,
    suppress_fit_logs=True,
)

display(df)

print("\n=== Aggregate (across seeds) ===")
cols = ["v_L1_final","A_Fro_final","A_MAE_final","mean_gap_to_oracle_end","mean_err_avg","vx_gap_to_oracle_end","vx_err_avg"]
display(df[cols].describe().T)

# =========================================================
# Quick plots to inspect variability across seeds
# (matplotlib default colors as requested)
# =========================================================
plt.figure(figsize=(8,3))
plt.plot(df["seed"], df["v_L1_final"], marker="o")
plt.xlabel("seed"); plt.ylabel("||v_hat - v_true||_1")
plt.title("Final centrality error vs seed")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8,3))
plt.plot(df["seed"], df["A_MAE_final"], marker="o")
plt.xlabel("seed"); plt.ylabel("MAE(A_hat, A_true)")
plt.title("Final adjacency MAE vs seed")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8,3))
plt.plot(df["seed"], df["mean_gap_to_oracle_end"], marker="o", label="|mean_learn(T)-mean_oracle(T)|")
plt.plot(df["seed"], df["vx_gap_to_oracle_end"], marker="o", label="|v_true^T x_learn(T)-v_true^T x_oracle(T)|")
plt.xlabel("seed"); plt.ylabel("end-gap")
plt.title("End-of-experiment trajectory gap vs seed")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()